<!-- R01 -->
# Ripristino dei risultati di riferimento
Computer Vision · Lezione 16. Eseguire R01–R08 in ordine dalla cartella recupero, oppure in Colab caricando lo ZIP studenti quando richiesto da R02. I tempi e il training del JSON provengono dalla corsa CPU consegnata. La nuova inferenza sottostante appartiene al dispositivo corrente. Questo percorso non completa i tre TODO del notebook studenti e non riprende l’ottimizzazione.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

In [ ]:
# R02
from pathlib import Path
import sys, os, json, copy, time, zipfile, importlib.util, subprocess, io
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import files
    root=Path('/content/ComputerVision_Lezione16')
    candidates=list(root.glob('**/supporto/detection_utils.py')) if root.exists() else []
    if not candidates:
        print('Carica Pacchetto_studenti_Lezione_16.zip. Il solo file ipynb non contiene dati o supporto.')
        upload=files.upload()
        zips=[name for name in upload if name.endswith('.zip')]
        if len(zips)!=1: raise RuntimeError('Caricare un solo ZIP studenti della lezione 16.')
        root.mkdir(exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(upload[zips[0]])) as z:
            for name in z.namelist():
                if not (root/name).resolve().is_relative_to(root.resolve()): raise ValueError('Percorso ZIP non sicuro')
            z.extractall(root)
        candidates=list(root.glob('**/supporto/detection_utils.py'))
    if len(candidates)!=1: raise RuntimeError('Pacchetto non riconosciuto: serve una sola cartella supporto.')
    BASE=candidates[0].parent.parent
    os.chdir(BASE/'recupero')
    if importlib.util.find_spec('pycocotools') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','--no-deps','pycocotools==2.0.11'])
else:
    BASE=Path.cwd().parent
    if not (BASE/'supporto/detection_utils.py').exists(): raise RuntimeError('Aprire dalla cartella recupero del pacchetto estratto.')
import numpy as np
import torch, torchvision
from PIL import Image
from torchvision.ops import nms
assert nms(torch.tensor([[0.,0.,2.,2.]]),torch.tensor([.9]),.5).tolist()==[0]
sys.path.insert(0,str(BASE/'supporto'))
import detection_utils as u
DEVICE=os.environ.get('CV_DEVICE','auto')  # auto sceglie CUDA disponibile, altrimenti CPU
ENV=u.setup(DEVICE);DEVICE=ENV['device']
OUT=Path.cwd()/'output_recupero';OUT.mkdir(exist_ok=True)
print(ENV)
print('Protocollo:',u.CONFIG)


In [ ]:
# R03
REC=BASE/'recupero'
provenance=u.prepare(BASE,download=True)
record=json.loads((REC/'metriche_finali.json').read_text())
manifest=json.loads((BASE/'manifest.json').read_text())
for name,digest in manifest['reference_files'].items():assert u.sha(REC/name)==digest
assert provenance==record['provenance']
print('Training di riferimento:',record['environment'])
print('Inferenza corrente:',ENV)

In [ ]:
# R04
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
model=fasterrcnn_mobilenet_v3_large_320_fpn(weights=u.WEIGHTS,progress=False,trainable_backbone_layers=0,box_score_thresh=.001,box_nms_thresh=.5,box_detections_per_img=100)
model.roi_heads.box_predictor=FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,2)
model.roi_heads.box_predictor.load_state_dict(torch.load(REC/'predictor_epoch3.pt',weights_only=True,map_location='cpu'))
model.to(DEVICE).eval()
print('Pesi COCO e predictor epoch3 caricati. Solo inferenza, nessuna soluzione del TODO congelamento.')

In [ ]:
# R05
file=BASE/'dati/PennFudanPed/PNGImages/FudanPed00001.png'
x=torch.from_numpy(np.array(Image.open(file).convert('RGB'))).permute(2,0,1).float()/255
with torch.no_grad():pred=model([x.to(DEVICE)])[0]
reference=torch.load(REC/'predictions.pt',weights_only=True)['epoch3'][0]
same_shapes=all(pred[k].shape==reference[k].shape for k in ['boxes','scores'])
error=max(float((pred[k].cpu()-reference[k]).abs().max()) for k in ['boxes','scores']) if same_shapes else None
print('Forma coincidente:',same_shapes,'errore massimo:',error)
if DEVICE=='cpu' and ENV['torch']==record['environment']['torch'] and ENV['torchvision']==record['environment']['torchvision']:
    assert same_shapes and error<1e-5
else:
    print('Versioni o dispositivo diversi: confronto diagnostico, nessuna promessa di identità numerica.')

In [ ]:
# R06
print('Metriche importate, non ricalcolate in questa VM:')
for k in ['AP','AP50','AP75','AP_small','AP_medium','AP_large']:
    print(k,record['epoch0'][k],record['epoch3'][k])
print('Tempo training CPU importato [s]:',record['training']['seconds'])
summary={'origin':'training e metriche CPU importati; inferenza singola nuova','reference_environment':record['environment'],'current_environment':ENV,'same_shapes':same_shapes,'max_error':error,'reference_AP':record['epoch3']['AP']}
(OUT/'verifica_recupero.json').write_text(json.dumps(summary,indent=2))

<!-- R07 -->
## Interpretazione dei risultati
Usare i valori importati per discutere la detection card. L’HTML mostra anche le figure senza richiedere dataset o kernel. Per tornare al percorso di training, riavviare il notebook studenti e completare C07, C12, C19. Il recupero non definisce le soluzioni dei TODO né lo stato completo di SGD. Dichiarare sempre la provenienza delle misure.

In [ ]:
# R08
archive=__import__('shutil').make_archive(str(Path.cwd()/'Verifica_recupero_Lezione16'),'zip',OUT)
print(archive)
if IN_COLAB:files.download(archive)